# Other Models - Price Only

In [2]:
import numpy as np
import pandas as pd

# core models
from catboost import CatBoostRegressor
from lightgbm import LGBMRegressor

# torch for NN
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import Dataset, DataLoader

In [3]:
# import data
train = pd.read_parquet('../data/model/final_train.parquet')
val = pd.read_parquet('../data/model/final_val.parquet')
test = pd.read_parquet('../data/model/final_test.parquet')

In [4]:
# check the min and max Date in each dataset
print("Train Date range:", train['Date'].min(), "to", train['Date'].max())
print("Validation Date range:", val['Date'].min(), "to", val['Date'].max())
print("Test Date range:", test['Date'].min(), "to", test['Date'].max())

Train Date range: 2016-01-04 00:00:00 to 2021-12-30 00:00:00
Validation Date range: 2022-01-03 00:00:00 to 2022-12-30 00:00:00
Test Date range: 2023-01-03 00:00:00 to 2023-12-29 00:00:00


In [5]:
X_train = train.loc[:, train.columns != "return_next_day"]
X_val = val.loc[:, val.columns != "return_next_day"]
X_test = test.loc[:, test.columns != "return_next_day"]
X_full = pd.concat([X_train, X_val, X_test], axis=0).reset_index(drop=True)

In [6]:
y_train = train[['Date', 'tic', 'return_next_day']]
y_val = val[['Date', 'tic', 'return_next_day']]
y_test = test[['Date', 'tic', 'return_next_day']]
y_full = pd.concat([y_train, y_val, y_test], axis=0).reset_index(drop=True)

In [7]:
def directional_accuracy(y_true, y_pred):
    return (np.sign(y_true) == np.sign(y_pred)).mean()

In [8]:
def add_rolling_mean_feature(df, group_col, value_col, window, feature_name, shift_before=True):
    """
    Rolling mean feature per group_col.

    If shift_before=True, shift by 1 so feature at time t only uses info before t.
    """
    df = df.copy()
    df[feature_name] = (
        df.groupby(group_col)[value_col]
          .transform(lambda x: x.rolling(window=window, min_periods=1).mean())
    )
    if shift_before:
        df[feature_name] = df.groupby(group_col)[feature_name].shift(1)
    return df


def add_ema_feature(df, group_col, value_col, span, feature_name, shift_before=True):
    """
    Exponential moving average feature per group_col.

    If shift_before=True, shift by 1 so there is no look-ahead.
    """
    df = df.copy()
    df[feature_name] = (
        df.groupby(group_col)[value_col]
          .transform(lambda x: x.ewm(span=span, adjust=False, min_periods=1).mean())
    )
    if shift_before:
        df[feature_name] = df.groupby(group_col)[feature_name].shift(1)
    return df

In [9]:
def compute_signal_features_from_Xfull(X_full):
    """
    X_full must have at least: Date, tic, Close.
    We compute ret internally and build all the signal features.

    Returns a DataFrame with:
        ['Date', 'tic',
         's4_dow_mean',
         's7_roll63_mean_return',
         'ema_ema_50', 'ema_ema_20', 'ema_ema_10',
         'b0',
         'ma_1m', 'ma_3d', 'ma_14d', 'ma_30d', 'ma_50d',
         'ma_2w', 'ma_2d', 'ma_1w',
         's3_month_of_year_mean']
    """
    df = X_full[['Date', 'tic', 'Close']].copy()
    df['Date'] = pd.to_datetime(df['Date'])
    df = df.sort_values(['tic', 'Date']).reset_index(drop=True)

    # --- realized daily return (today vs yesterday) ---
    df['ret'] = df.groupby('tic')['Close'].pct_change()

    # --- b0: previous day's return per ticker ---
    df['b0'] = df.groupby('tic')['ret'].shift(1)

    # --- s4_dow_mean: prior day's cross-sectional mean ret (market-like) ---
    # cross-sectional mean ret per day
    df['dow_mean_ret'] = df.groupby('Date')['ret'].transform('mean')
    # use previous day's dow_mean_ret for each ticker to avoid look-ahead
    df['s4_dow_mean'] = df.groupby('tic')['dow_mean_ret'].shift(1)

    # --- s7_roll63_mean_return: 63-day rolling mean of ret per ticker ---
    df = add_rolling_mean_feature(
        df,
        group_col='tic',
        value_col='ret',
        window=63,
        feature_name='s7_roll63_mean_return',
        shift_before=True,
    )

    # --- EMA features on ret per ticker ---
    df = add_ema_feature(
        df,
        group_col='tic',
        value_col='ret',
        span=50,
        feature_name='ema_ema_50',
        shift_before=True,
    )
    df = add_ema_feature(
        df,
        group_col='tic',
        value_col='ret',
        span=20,
        feature_name='ema_ema_20',
        shift_before=True,
    )
    df = add_ema_feature(
        df,
        group_col='tic',
        value_col='ret',
        span=10,
        feature_name='ema_ema_10',
        shift_before=True,
    )

    # --- Moving-average features on ret per ticker ---
    ma_windows = {
        'ma_2d': 2,
        'ma_3d': 3,
        'ma_14d': 14,
        'ma_1w': 5,   # ~1 week
        'ma_2w': 10,  # ~2 weeks
        'ma_1m': 21,  # ~1 month
        'ma_30d': 30,
        'ma_50d': 50,
    }

    for label, w in ma_windows.items():
        df = add_rolling_mean_feature(
            df,
            group_col='tic',
            value_col='ret',
            window=w,
            feature_name=label,
            shift_before=True,
        )

    # --- s3_month_of_year_mean: expanding month-of-year mean ret per (tic, month) ---
    df['month'] = df['Date'].dt.month
    df = df.sort_values(['tic', 'month', 'Date']).reset_index(drop=True)

    def _expanding_month_mean(s):
        # expanding mean of past values only
        return s.shift(1).expanding(min_periods=1).mean()

    df['s3_month_of_year_mean'] = (
        df.groupby(['tic', 'month'])['ret'].transform(_expanding_month_mean)
    )

    # Sort back by tic, Date
    df = df.sort_values(['tic', 'Date']).reset_index(drop=True)

    feature_cols = [
        's4_dow_mean',
        's7_roll63_mean_return',
        'ema_ema_50',
        'ema_ema_20',
        'ema_ema_10',
        'b0',
        'ma_1m',
        'ma_3d',
        'ma_14d',
        'ma_30d',
        'ma_50d',
        'ma_2w',
        'ma_2d',
        'ma_1w',
        's3_month_of_year_mean',
    ]

    # Fill NaNs in features with 0 (no historical info yet → neutral signal)
    df[feature_cols] = df[feature_cols].fillna(0.0)

    features_df = df[['Date', 'tic'] + feature_cols].copy()
    return features_df

In [10]:
features_full = compute_signal_features_from_Xfull(X_full)
features_full.head()

,Date,tic,s4_dow_mean,s7_roll63_mean_return,ema_ema_50,ema_ema_20,ema_ema_10,b0,ma_1m,ma_3d,ma_14d,ma_30d,ma_50d,ma_2w,ma_2d,ma_1w,s3_month_of_year_mean
0,2016-01-04,A,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2016-01-05,A,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2016-01-06,A,0.002172,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440
3,2016-01-07,A,-0.015380,0.000499,-0.003131,-0.002690,-0.002008,0.004439,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499
4,2016-01-08,A,-0.024183,-0.013825,-0.004674,-0.006479,-0.009365,-0.042474,-0.013825,-0.013825,-0.013825,-0.013825,-0.013825,-0.013825,-0.019018,-0.013825,-0.013825


In [11]:
# Augment design matrices with signal features
X_train_aug = X_train.merge(features_full, on=['Date', 'tic'], how='left')
X_val_aug   = X_val.merge(features_full,   on=['Date', 'tic'], how='left')
X_test_aug  = X_test.merge(features_full,  on=['Date', 'tic'], how='left')

# Optional sanity checks
print("X_train:", X_train.shape, "->", X_train_aug.shape)
print("X_val:  ", X_val.shape,   "->", X_val_aug.shape)
print("X_test: ", X_test.shape,  "->", X_test_aug.shape)

# Check for missing values in new features (should be minor if at all)
X_train_aug.isna().mean().sort_values().tail(15)

X_train: (729659, 108) -> (729659, 123)
X_val:   (124747, 108) -> (124747, 123)
X_test:  (124727, 108) -> (124727, 123)


pca_emb_0                0.0
news_count               0.0
sum_sentiment            0.0
min_sentiment            0.0
max_sentiment            0.0
mean_sentiment           0.0
yield_spread_10y_2y      0.0
sp500                    0.0
vix                      0.0
aaa_yield                0.0
t3m                      0.0
t2y                      0.0
t10y                     0.0
retail_sales             0.0
s3_month_of_year_mean    0.0
dtype: float64

In [12]:
feature_cols = [
    's4_dow_mean',
    's7_roll63_mean_return',
    'ema_ema_50',
    'ema_ema_20',
    'ema_ema_10',
    'b0',
    'ma_1m',
    'ma_3d',
    'ma_14d',
    'ma_30d',
    'ma_50d',
    'ma_2w',
    'ma_2d',
    'ma_1w',
    's3_month_of_year_mean',
]

In [13]:
# columns we want to keep
keep_cols = ['Date', 'tic', 'Close'] + feature_cols

X_train_sig = X_train_aug[keep_cols].copy()
X_val_sig   = X_val_aug[keep_cols].copy()
X_test_sig  = X_test_aug[keep_cols].copy()

In [14]:
X_train_sig.head()

,Date,tic,Close,s4_dow_mean,s7_roll63_mean_return,ema_ema_50,ema_ema_20,ema_ema_10,b0,ma_1m,ma_3d,ma_14d,ma_30d,ma_50d,ma_2w,ma_2d,ma_1w,s3_month_of_year_mean
0,2016-01-04,A,37.636356,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2016-01-05,A,37.506878,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,2016-01-06,A,37.673370,0.002172,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440,-0.003440
3,2016-01-07,A,36.073215,-0.015380,0.000499,-0.003131,-0.002690,-0.002008,0.004439,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499,0.000499
4,2016-01-08,A,35.693970,-0.024183,-0.013825,-0.004674,-0.006479,-0.009365,-0.042474,-0.013825,-0.013825,-0.013825,-0.013825,-0.013825,-0.013825,-0.019018,-0.013825,-0.013825


In [15]:
# Features
X_train_m = X_train_sig.drop(columns=["Date", "tic"])
X_val_m   = X_val_sig.drop(columns=["Date", "tic"])
X_test_m  = X_test_sig.drop(columns=["Date", "tic"])

# Targets
y_train_vec = y_train["return_next_day"]
y_val_vec   = y_val["return_next_day"]
y_test_vec  = y_test["return_next_day"]

In [16]:
# optional but helpful: cast to float32
X_train_m_32 = X_train_m.astype("float32")
X_val_m_32   = X_val_m.astype("float32")
X_test_m_32  = X_test_m.astype("float32")

y_train_vec = y_train_vec.astype("float32")
y_val_vec   = y_val_vec.astype("float32")
y_test_vec  = y_test_vec.astype("float32")

In [17]:
sp500_info = pd.read_csv('../data/sp500_companies.csv')
sp500_info.head()

,ticker,company_name,sector,subsector,cik
0,MMM,3M,Industrials,Industrial Conglomerates,66740
1,AOS,A. O. Smith,Industrials,Building Products,91142
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,1800
3,ABBV,AbbVie,Health Care,Biotechnology,1551152
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,1467373


In [18]:
def add_time_and_volume_features(df):
    df = df.copy()
    df["Date"] = pd.to_datetime(df["Date"])
    df["day_of_week"] = df["Date"].dt.dayofweek
    df["month"]       = df["Date"].dt.month
    df["log_volume"]  = np.log1p(df["Volume"])
    return df

In [19]:
# Create augmented versions instead
X_train_tv = add_time_and_volume_features(X_train)
X_val_tv   = add_time_and_volume_features(X_val)
X_test_tv  = add_time_and_volume_features(X_test)

In [20]:
# 1. Prepare sector/subsector lookup
sector_info = (
    sp500_info[["ticker", "sector", "subsector"]]
    .rename(columns={"ticker": "tic"})
)

# 2. Add sector + subsector to each split (starting from your time/volume-augmented versions)
X_train_tv_sec = X_train_tv.merge(sector_info, on="tic", how="left")
X_val_tv_sec   = X_val_tv.merge(sector_info,   on="tic", how="left")
X_test_tv_sec  = X_test_tv.merge(sector_info,  on="tic", how="left")

X_train_tv_sec.head()

,Date,tic,Open,High,Low,Close,Volume,sales_growth_qoq,sales_growth_ttm,asset_growth,...,pca_emb_59,pca_emb_60,pca_emb_61,pca_emb_62,pca_emb_63,day_of_week,month,log_volume,sector,subsector
0,2016-01-04,A,37.978592,38.098833,37.312624,37.636356,3287300,0.02071,-0.00247,-0.309482,...,0.000000,0.000000,0.000000,0.000000,0.000000,0,1,15.005577,Health Care,Life Sciences Tools & Services
1,2016-01-05,A,37.673370,37.876861,37.312638,37.506878,2587200,0.02071,-0.00247,-0.309482,...,0.000000,0.000000,0.000000,0.000000,0.000000,1,1,14.766087,Health Care,Life Sciences Tools & Services
2,2016-01-06,A,37.220145,37.913860,37.044401,37.673370,2103600,0.02071,-0.00247,-0.309482,...,-0.750782,-0.323059,0.027356,-0.609138,-0.380578,2,1,14.559161,Health Care,Life Sciences Tools & Services
3,2016-01-07,A,37.127663,37.136914,35.897475,36.073215,3504300,0.02071,-0.00247,-0.309482,...,-0.062964,-0.121446,0.122892,-0.254481,0.042727,3,1,15.069502,Health Care,Life Sciences Tools & Services
4,2016-01-08,A,36.276692,36.729917,35.582976,35.693970,3736700,0.02071,-0.00247,-0.309482,...,0.000000,0.000000,0.000000,0.000000,0.000000,4,1,15.133714,Health Care,Life Sciences Tools & Services


In [21]:
cat_cols = ["day_of_week", "month", "sector", "subsector"]

# 1) One-hot encode each split, overwriting the original variables
X_train_tv_sec = pd.get_dummies(X_train_tv_sec, columns=cat_cols, drop_first=True)
X_val_tv_sec   = pd.get_dummies(X_val_tv_sec,   columns=cat_cols, drop_first=True)
X_test_tv_sec  = pd.get_dummies(X_test_tv_sec,  columns=cat_cols, drop_first=True)

# 2) Align val/test columns to train's columns (any missing dummies → 0)
train_cols = X_train_tv_sec.columns

X_val_tv_sec  = X_val_tv_sec.reindex(columns=train_cols, fill_value=0)
X_test_tv_sec = X_test_tv_sec.reindex(columns=train_cols, fill_value=0)

In [22]:
def select_stock_structure_features(df):
    base_cols = ["Date", "tic", "Close", "log_volume"]
    
    dow_cols      = [c for c in df.columns if c.startswith("day_of_week_")]
    month_cols    = [c for c in df.columns if c.startswith("month_")]
    sector_cols   = [c for c in df.columns if c.startswith("sector_")]
    subsector_cols= [c for c in df.columns if c.startswith("subsector_")]
    
    keep_cols = base_cols + dow_cols + month_cols + sector_cols + subsector_cols
    return df[keep_cols].copy()

X_train_core = select_stock_structure_features(X_train_tv_sec)
X_val_core   = select_stock_structure_features(X_val_tv_sec)
X_test_core  = select_stock_structure_features(X_test_tv_sec)

In [23]:
X_train_core.head()

,Date,tic,Close,log_volume,day_of_week_1,day_of_week_2,day_of_week_3,day_of_week_4,month_2,month_3,...,subsector_Systems Software,subsector_Technology Distributors,"subsector_Technology Hardware, Storage & Peripherals",subsector_Telecom Tower REITs,subsector_Timber REITs,subsector_Tobacco,subsector_Trading Companies & Distributors,subsector_Transaction & Payment Processing Services,subsector_Water Utilities,subsector_Wireless Telecommunication Services
0,2016-01-04,A,37.636356,15.005577,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,2016-01-05,A,37.506878,14.766087,True,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,2016-01-06,A,37.673370,14.559161,False,True,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,2016-01-07,A,36.073215,15.069502,False,False,True,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,2016-01-08,A,35.693970,15.133714,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,False,False


In [24]:
# Features to use for modeling = all columns except Date, tic
core_feature_cols = [c for c in X_train_core.columns if c not in ["Date", "tic"]]

X_train_core_m = X_train_core[core_feature_cols]
X_val_core_m   = X_val_core[core_feature_cols]
X_test_core_m  = X_test_core[core_feature_cols]

In [25]:
X_train_core_32 = X_train_core_m.astype("float32")
X_val_core_32   = X_val_core_m.astype("float32")
X_test_core_32  = X_test_core_m.astype("float32")

y_train_vec = y_train_vec.astype("float32")
y_val_vec   = y_val_vec.astype("float32")
y_test_vec  = y_test_vec.astype("float32")

In [26]:
# Drop Close from core so we keep the one from X_*_sig
X_train_sig_core = X_train_sig.merge(
    X_train_core.drop(columns=["Close"]),
    on=["Date", "tic"],
    how="left",
)

X_val_sig_core = X_val_sig.merge(
    X_val_core.drop(columns=["Close"]),
    on=["Date", "tic"],
    how="left",
)

X_test_sig_core = X_test_sig.merge(
    X_test_core.drop(columns=["Close"]),
    on=["Date", "tic"],
    how="left",
)

In [27]:
feature_cols_all = [c for c in X_train_sig_core.columns if c not in ["Date", "tic"]]

X_train_sig_core_m = X_train_sig_core[feature_cols_all]
X_val_sig_core_m   = X_val_sig_core[feature_cols_all]
X_test_sig_core_m  = X_test_sig_core[feature_cols_all]

In [28]:
X_train_sig_core_32 = X_train_sig_core_m.astype("float32")
X_val_sig_core_32   = X_val_sig_core_m.astype("float32")
X_test_sig_core_32  = X_test_sig_core_m.astype("float32")

y_train_vec = y_train_vec.astype("float32")
y_val_vec   = y_val_vec.astype("float32")
y_test_vec  = y_test_vec.astype("float32")

In [29]:
sig_cols = feature_cols                                # price-signal features
vol_cols = ["log_volume"]
dow_cols = [c for c in X_train_sig_core_32.columns if c.startswith("day_of_week_")]
month_cols = [c for c in X_train_sig_core_32.columns if c.startswith("month_")]
sector_cols = [c for c in X_train_sig_core_32.columns if c.startswith("sector_")]
subsector_cols = [c for c in X_train_sig_core_32.columns if c.startswith("subsector_")]

In [30]:
# Start from the same df used for your catboost experiments
# (this should be the one that already has the one-hot cols)
X_train_sig_core   # <- whatever you've been using

# Identify the DOW / MONTH / SECTOR columns
dow_cols = [c for c in X_train_sig_core.columns if c.startswith("day_of_week_")]
month_cols = [c for c in X_train_sig_core.columns if c.startswith("month_")]
sector_cols = [c for c in X_train_sig_core.columns if c.startswith("sector_")]

dms_cols = dow_cols + month_cols + sector_cols

print("Num DOW cols:   ", len(dow_cols))
print("Num MONTH cols: ", len(month_cols))
print("Num SECTOR cols:", len(sector_cols))
print("Total DMS cols: ", len(dms_cols))

# Build train/val/test subsets
X_train_dms = X_train_sig_core[dms_cols].copy()
X_val_dms   = X_val_sig_core[dms_cols].copy()
X_test_dms  = X_test_sig_core[dms_cols].copy()

Num DOW cols:    4
Num MONTH cols:  11
Num SECTOR cols: 10
Total DMS cols:  25


#### Model 5 - Add ticker fixed effects via embeddings **DON'T RUN AGAIN**

We experimented with a neural network with ticker embeddings. Despite early stopping and sign-aware training, its validation directional accuracy (~0.50) remained below that of our tuned gradient-boosted trees (~0.52), so we kept XGBoost as our primary model.

**Will NOT run again. Just noting that the results are worst than XGBoost**

### Neural Network Experiments (Not Used)

Initial experiments with a ticker-embedding neural network were conducted but not included in the final pipeline due to:
- High computational cost and unstable training on the full panel
- Directional accuracy comparable to or worse than tuned tree-based models

Subsequent modeling focuses on tree-based methods (XGBoost, CatBoost, LightGBM) on the DOW+MONTH+SECTOR feature set.

In [30]:
feature_cols_all = [c for c in X_train_sig_core.columns if c not in ["Date", "tic"]]
X_train_sig_core_m = X_train_sig_core[feature_cols_all]
X_val_sig_core_m   = X_val_sig_core[feature_cols_all]
X_test_sig_core_m  = X_test_sig_core[feature_cols_all]

In [31]:
# ==============================
# 1. Ticker vocabulary (train only)
# ==============================
all_tickers_train = X_train["tic"].unique()   # or X_train_sig_core["tic"]
ticker2id = {tic: i for i, tic in enumerate(sorted(all_tickers_train))}
num_tickers = len(ticker2id)
print("Num tickers:", num_tickers)

def tic_to_ids(tic_series):
    """Map ticker strings to integer IDs using train-based vocab."""
    return tic_series.map(ticker2id).values

# ==============================
# 2. Numeric feature matrices: DOW + MONTH + SECTOR only (DMS)
# ==============================
X_train_num = X_train_dms.values.astype("float32")
X_val_num   = X_val_dms.values.astype("float32")
X_test_num  = X_test_dms.values.astype("float32")

print("X_train_num shape:", X_train_num.shape)
print("X_val_num shape:  ", X_val_num.shape)
print("X_test_num shape: ", X_test_num.shape)

# ==============================
# 3. Ticker ID vectors for each split
# ==============================
train_tic_ids = tic_to_ids(X_train["tic"]).astype("int64")
val_tic_ids   = tic_to_ids(X_val["tic"]).astype("int64")
test_tic_ids  = tic_to_ids(X_test["tic"]).astype("int64")

# ==============================
# 4. Targets as float32
# ==============================
# y_*_vec is assumed to be 1D numpy array or Series
y_train_t = y_train_vec.astype("float32")
y_val_t   = y_val_vec.astype("float32")
y_test_t  = y_test_vec.astype("float32")

Num tickers: 496
X_train_num shape: (729659, 25)
X_val_num shape:   (124747, 25)
X_test_num shape:  (124727, 25)


/var/folders/c2/pprfn7j56vs360hbrz08yqkh0000gn/T/ipykernel_76104/942094126.py:28: RuntimeWarning: invalid value encountered in cast
  val_tic_ids   = tic_to_ids(X_val["tic"]).astype("int64")
/var/folders/c2/pprfn7j56vs360hbrz08yqkh0000gn/T/ipykernel_76104/942094126.py:29: RuntimeWarning: invalid value encountered in cast
  test_tic_ids  = tic_to_ids(X_test["tic"]).astype("int64")


In [32]:
from torch.utils.data import Dataset, DataLoader
import torch

# ==============================
# 1) Convert DMS feature frames to numpy (float32)
# ==============================
X_train_dms_num = X_train_dms.values.astype("float32")
X_val_dms_num   = X_val_dms.values.astype("float32")
X_test_dms_num  = X_test_dms.values.astype("float32")

# ==============================
# 2) PyTorch Dataset for DMS + ticker embedding
# ==============================
class StockDataset(Dataset):
    def __init__(self, X_num, tic_ids, y):
        self.X_num = X_num              # numpy array, shape (N, num_features_dms)
        self.tic_ids = tic_ids          # numpy array, shape (N,)
        self.y = y                      # numpy array, shape (N,)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "x_num": torch.from_numpy(self.X_num[idx]),                     # float32
            "tic_id": torch.tensor(self.tic_ids[idx], dtype=torch.long),    # int64
            "y": torch.tensor(self.y[idx], dtype=torch.float32),            # float32
        }

# ==============================
# 3) Train/val/test datasets + loaders (DMS only)
# ==============================
train_ds = StockDataset(X_train_dms_num, train_tic_ids, y_train_t)
val_ds   = StockDataset(X_val_dms_num,   val_tic_ids,   y_val_t)
test_ds  = StockDataset(X_test_dms_num,  test_tic_ids,  y_test_t)

train_loader = DataLoader(train_ds, batch_size=2048, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=4096, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=2048, shuffle=False)

In [33]:
import torch.nn as nn
import torch.nn.functional as F

class TickerEmbeddingModel(nn.Module):
    def __init__(
        self,
        num_tickers,
        num_features,
        emb_dim=16,
        num_hidden_dim=64,
        joint_hidden_dim=64,
        joint_hidden_dim2=32,
        dropout_p=0.2,
    ):
        super().__init__()

        # --- Ticker embedding tower ---
        self.ticker_emb = nn.Embedding(num_embeddings=num_tickers, embedding_dim=emb_dim)
        self.tic_fc = nn.Linear(emb_dim, emb_dim)  # small projection

        # --- Numeric feature tower ---
        self.num_fc1 = nn.Linear(num_features, num_hidden_dim)
        self.num_fc2 = nn.Linear(num_hidden_dim, num_hidden_dim)

        # --- Joint tower ---
        joint_in_dim = num_hidden_dim + emb_dim
        self.joint_fc1 = nn.Linear(joint_in_dim, joint_hidden_dim)
        self.joint_fc2 = nn.Linear(joint_hidden_dim, joint_hidden_dim2)

        self.out = nn.Linear(joint_hidden_dim2, 1)

        self.dropout = nn.Dropout(dropout_p)
        self.bn_num = nn.BatchNorm1d(num_hidden_dim)

    def forward(self, x_num, tic_id):
        # Numeric tower
        x_num = F.relu(self.num_fc1(x_num))
        x_num = self.bn_num(x_num)
        x_num = F.relu(self.num_fc2(x_num))
        x_num = self.dropout(x_num)

        # Ticker tower
        emb = self.ticker_emb(tic_id)
        emb = F.relu(self.tic_fc(emb))

        # Joint tower
        x = torch.cat([x_num, emb], dim=1)
        x = F.relu(self.joint_fc1(x))
        x = self.dropout(x)
        x = F.relu(self.joint_fc2(x))
        x = self.dropout(x)

        out = self.out(x).squeeze(-1)
        return out

In [34]:
num_features = X_train_num.shape[1]

model = TickerEmbeddingModel(
    num_tickers=num_tickers,
    num_features=num_features,
    emb_dim=16,
    num_hidden_dim=64,
    joint_hidden_dim=64,
    joint_hidden_dim2=32,
    dropout_p=0.2,
)

In [35]:
len(train_ds), len(train_loader)

(729659, 357)

In [36]:
def sign_aware_loss(pred, y, lambda_sign=0.1):
    mse = F.mse_loss(pred, y)

    # sign mismatch penalty: 1 if wrong sign, 0 if correct
    sign_mismatch = (torch.sign(pred) != torch.sign(y)).float()

    # average mismatch (fraction wrong)
    sign_penalty = sign_mismatch.mean()

    return mse + lambda_sign * sign_penalty

In [37]:
import torch
import torch.nn as nn
from torch.optim import Adam
import torch.nn.functional as F

# --- loss: MSE + sign penalty ---
def sign_aware_loss(pred, y, lambda_sign=0.1):
    mse = F.mse_loss(pred, y)
    sign_mismatch = (torch.sign(pred) != torch.sign(y)).float()
    sign_penalty = sign_mismatch.mean()
    return mse + lambda_sign * sign_penalty

def directional_accuracy_torch(y_true, y_pred):
    """
    y_true, y_pred: torch tensors, shape (batch,)
    Returns DA as a Python float.
    """
    return (torch.sign(y_true) == torch.sign(y_pred)).float().mean().item()

# --- optimizer & loss ---
optimizer = Adam(model.parameters(), lr=5e-4, weight_decay=1e-4)
loss_fn = lambda pred, y: sign_aware_loss(pred, y, lambda_sign=0.1)

# -------------------------
# Training loop with early stopping on Val DA
# -------------------------
num_epochs = 25
patience   = 10
best_val_da = -1.0
patience_counter = 0
best_state = None

for epoch in range(num_epochs):
    # ---- Train ----
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        x_num  = batch["x_num"]   # already CPU tensors
        tic_id = batch["tic_id"]
        y      = batch["y"]

        optimizer.zero_grad()
        pred = model(x_num, tic_id)
        loss = loss_fn(pred, y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * len(y)

    train_loss /= len(train_ds)

    # ---- Validation ----
    model.eval()
    val_loss = 0.0
    val_da_sum = 0.0
    n_val = 0

    with torch.no_grad():
        for batch in val_loader:
            x_num  = batch["x_num"]
            tic_id = batch["tic_id"]
            y      = batch["y"]

            pred = model(x_num, tic_id)
            loss = loss_fn(pred, y)

            val_loss += loss.item() * len(y)
            batch_da = directional_accuracy_torch(y, pred)
            val_da_sum += batch_da * len(y)
            n_val += len(y)

    val_loss /= n_val
    val_da = val_da_sum / n_val

    print(
        f"Epoch {epoch+1:02d} | "
        f"train_loss={train_loss:.6f} | "
        f"val_loss={val_loss:.6f} | "
        f"val_DA={val_da:.4f}"
    )

    # ---- Early stopping on Val DA ----
    if val_da > best_val_da + 1e-4:
        best_val_da = val_da
        best_state = model.state_dict()
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

# Restore best model
if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Restored best model with Val DA = {best_val_da:.4f}")

: 

In [ ]:
def predict_loader(model, loader):
    model.eval()
    preds = []
    ys    = []
    with torch.no_grad():
        for batch in loader:
            x_num = batch["x_num"].to(device)
            tic_id = batch["tic_id"].to(device)
            y = batch["y"].to(device)

            pred = model(x_num, tic_id)
            preds.append(pred.cpu().numpy())
            ys.append(y.cpu().numpy())

    preds = np.concatenate(preds)
    ys    = np.concatenate(ys)
    return ys, preds

y_val_true_emb, y_val_pred_emb   = predict_loader(model, val_loader)
y_test_true_emb, y_test_pred_emb = predict_loader(model, test_loader)

val_DA_emb  = (np.sign(y_val_pred_emb)  == np.sign(y_val_true_emb)).mean()
test_DA_emb = (np.sign(y_test_pred_emb) == np.sign(y_test_true_emb)).mean()

print("Ticker-Embedding Model:")
print("Val DA: ", val_DA_emb)
print("Test DA:", test_DA_emb)

Ticker-Embedding Model:
Val DA:  0.4930539411769421
Test DA: 0.5198553641152277


#### Model 6 - CAT boost

In [29]:
from catboost import CatBoostRegressor

# 1. Choose only numeric columns (drop Date and tic)
num_cols = [c for c in X_train_sig_core.columns if c not in ["Date", "tic"]]

X_train_cb = X_train_sig_core[num_cols]
X_val_cb   = X_val_sig_core[num_cols]
X_test_cb  = X_test_sig_core[num_cols]

# 2. Define CatBoost model
cat_model = CatBoostRegressor(
    depth=6,
    learning_rate=0.03,
    iterations=800,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

# 3. Fit on train → validate on val
cat_model.fit(
    X_train_cb,
    y_train_vec,
    eval_set=(X_val_cb, y_val_vec),
    verbose=False
)

# 4. Predictions + DA
pred_val_cat  = cat_model.predict(X_val_cb)
pred_test_cat = cat_model.predict(X_test_cb)

val_DA_cat  = directional_accuracy(y_val_vec,  pred_val_cat)
test_DA_cat = directional_accuracy(y_test_vec, pred_test_cat)

print("CatBoost (sig + core, numeric only):")
print("Val DA: ", val_DA_cat)
print("Test DA:", test_DA_cat)

CatBoost (sig + core, numeric only):
Val DA:  0.4929096491298388
Test DA: 0.5198553641152277


In [30]:
from catboost import CatBoostRegressor

# Use all columns except Date (keep tic as categorical)
feature_cols_cb = [c for c in X_train_sig_core.columns if c != "Date"]

X_train_cb = X_train_sig_core[feature_cols_cb]
X_val_cb   = X_val_sig_core[feature_cols_cb]
X_test_cb  = X_test_sig_core[feature_cols_cb]

# Cat features: columns that are strings / categories
cat_features = [feature_cols_cb.index("tic")]  # index of 'tic' in feature_cols_cb

cat_model = CatBoostRegressor(
    depth=6,
    learning_rate=0.03,
    iterations=800,
    loss_function="RMSE",
    random_seed=42,
    verbose=False
)

cat_model.fit(
    X_train_cb,
    y_train_vec,
    eval_set=(X_val_cb, y_val_vec),
    cat_features=cat_features,
    verbose=False
)

pred_val_cat  = cat_model.predict(X_val_cb)
pred_test_cat = cat_model.predict(X_test_cb)

val_DA_cat  = directional_accuracy(y_val_vec,  pred_val_cat)
test_DA_cat = directional_accuracy(y_test_vec, pred_test_cat)

print("CatBoost (tic as categorical):")
print("Val DA: ", val_DA_cat)
print("Test DA:", test_DA_cat)

CatBoost (tic as categorical):
Val DA:  0.4917312640784949
Test DA: 0.5181797044745725


In [31]:
import numpy as np
from catboost import CatBoostRegressor
from itertools import product

# ----- 1. Define a narrower, more reasonable search space -----
param_grid_cb = {
    "depth":            [4, 6, 8],
    "learning_rate":    [0.02, 0.03, 0.05],
    "l2_leaf_reg":      [1, 3, 5],
    "bagging_temperature": [0, 1, 5],
}

all_combos = list(product(
    param_grid_cb["depth"],
    param_grid_cb["learning_rate"],
    param_grid_cb["l2_leaf_reg"],
    param_grid_cb["bagging_temperature"],
))

# Randomly sample a subset of combos to try
n_samples = min(40, len(all_combos))  # <= 40 configs
np.random.seed(42)
sample_indices = np.random.choice(len(all_combos), size=n_samples, replace=False)
sampled_combos = [all_combos[i] for i in sample_indices]

# ----- 2. Prepare data & cat feature index -----
feature_cols_cb = [c for c in X_train_sig_core.columns if c != "Date"]
X_train_cb = X_train_sig_core[feature_cols_cb]
X_val_cb   = X_val_sig_core[feature_cols_cb]
X_test_cb  = X_test_sig_core[feature_cols_cb]

cat_features = [feature_cols_cb.index("tic")]  # index of 'tic' in feature_cols_cb

# ----- 3. Random search with early stopping -----
results_cb = []

for depth, lr, l2, bt in sampled_combos:
    print(f"Training CatBoost: depth={depth}, lr={lr}, l2={l2}, bt={bt}")

    model = CatBoostRegressor(
        depth=depth,
        learning_rate=lr,
        l2_leaf_reg=l2,
        bagging_temperature=bt,
        iterations=2000,            # large cap
        loss_function="RMSE",
        random_seed=42,
        verbose=False,
        thread_count=-1,
    )

    model.fit(
        X_train_cb, y_train_vec,
        eval_set=(X_val_cb, y_val_vec),
        cat_features=cat_features,
        use_best_model=True,        # keep best iteration
        early_stopping_rounds=100,  # stop if no improvement
        verbose=False,
    )

    pred_val = model.predict(X_val_cb)
    val_da   = directional_accuracy(y_val_vec, pred_val)

    results_cb.append({
        "depth": depth,
        "learning_rate": lr,
        "l2_leaf_reg": l2,
        "bagging_temperature": bt,
        "val_DA": val_da,
        "best_iteration": model.get_best_iteration(),
    })

results_cb_df = pd.DataFrame(results_cb).sort_values("val_DA", ascending=False)
results_cb_df.head()

Training CatBoost: depth=6, lr=0.02, l2=3, bt=0
Training CatBoost: depth=4, lr=0.02, l2=1, bt=0
Training CatBoost: depth=4, lr=0.05, l2=3, bt=1
Training CatBoost: depth=6, lr=0.02, l2=3, bt=1
Training CatBoost: depth=4, lr=0.05, l2=1, bt=0
Training CatBoost: depth=6, lr=0.02, l2=1, bt=1
Training CatBoost: depth=4, lr=0.03, l2=1, bt=1
Training CatBoost: depth=8, lr=0.03, l2=5, bt=1
Training CatBoost: depth=4, lr=0.02, l2=3, bt=1
Training CatBoost: depth=4, lr=0.03, l2=3, bt=0
Training CatBoost: depth=6, lr=0.05, l2=3, bt=1
Training CatBoost: depth=6, lr=0.02, l2=5, bt=0
Training CatBoost: depth=8, lr=0.03, l2=3, bt=1
Training CatBoost: depth=6, lr=0.02, l2=5, bt=5
Training CatBoost: depth=8, lr=0.03, l2=3, bt=5
Training CatBoost: depth=6, lr=0.05, l2=1, bt=0
Training CatBoost: depth=8, lr=0.05, l2=1, bt=1
Training CatBoost: depth=8, lr=0.02, l2=5, bt=1
Training CatBoost: depth=8, lr=0.02, l2=1, bt=1
Training CatBoost: depth=6, lr=0.03, l2=3, bt=1
Training CatBoost: depth=4, lr=0.03, l2=

,depth,learning_rate,l2_leaf_reg,bagging_temperature,val_DA,best_iteration
17,8,0.02,5,1,0.504333,42
25,8,0.02,5,5,0.504333,42
30,8,0.02,1,5,0.504028,52
27,8,0.02,1,0,0.504028,52
18,8,0.02,1,1,0.504028,52


In [ ]:
# Row with best validation DA
best_cb = results_cb_df.iloc[0]

# Keep only the actual CatBoost hyperparameters
best_cb_params = {
    "depth":             int(best_cb["depth"]),
    "learning_rate":     float(best_cb["learning_rate"]),
    "l2_leaf_reg":       float(best_cb["l2_leaf_reg"]),
    "bagging_temperature": float(best_cb["bagging_temperature"]),
    # use best_iteration as the number of trees
    "iterations":        int(best_cb["best_iteration"]),
    "loss_function":     "RMSE",
    "random_seed":       42
}

best_cb_model = CatBoostRegressor(
    **best_cb_params,
    verbose=False
)

# still pass cat_features in .fit, not in __init__
cat_idx = feature_cols_cb.index("tic")

# Option A: fit on TRAIN only, evaluate on VAL (pure eval)
best_cb_model.fit(
    X_train_cb,
    y_train_vec,
    cat_features=[cat_idx]
)

pred_val_cb  = best_cb_model.predict(X_val_cb)
pred_test_cb = best_cb_model.predict(X_test_cb)

print("CatBoost (retuned with best params):")
print("Val DA: ", directional_accuracy(y_val_vec,  pred_val_cb))
print("Test DA:", directional_accuracy(y_test_vec, pred_test_cb))

CatBoost (retuned with best params):
Val DA:  0.5023447457654292
Test DA: 0.5172256207557305


In [40]:
sig_cols = feature_cols                                # price-signal features
vol_cols = ["log_volume"]
dow_cols = [c for c in X_train_sig_core_32.columns if c.startswith("day_of_week_")]
month_cols = [c for c in X_train_sig_core_32.columns if c.startswith("month_")]
sector_cols = [c for c in X_train_sig_core_32.columns if c.startswith("sector_")]
subsector_cols = [c for c in X_train_sig_core_32.columns if c.startswith("subsector_")]

# Structural features we always keep
struct_cols = dow_cols + month_cols + sector_cols 

In [45]:
ranked_features = [
    "ma_1m",
    "s4_dow_mean",
    "ma_2d",
    "ma_14d",
    "ma_1w",
    "b0",
    "ma_2w",
    "ma_30d",
    "ema_ema_20",
    "ema_ema_50",
    "ma_3d",
    "ema_ema_10",
    "s7_roll63_mean_return",
    "ma_50d",
    "s3_month_of_year_mean",
    "Close",
]

top1_signal_cols = [ranked_features[0]]

In [46]:
core_cols = X_train_core_32.columns
sig_core_cols = X_train_sig_core_32.columns

In [47]:
feature_sets = {
    "core_struct": core_cols,                          # the set used in model 0
    "struct_plus_top1": struct_cols + top1_signal_cols, # model 8
    "dow_month_sector": dow_cols + month_cols + sector_cols,  # model 7
    "signal_plus_struct": sig_core_cols      # model 6
}

In [48]:
from catboost import CatBoostRegressor

cat_results = []

for name, cols in feature_sets.items():
    print(f"\n=== CatBoost on feature set: {name} ===")
    print(f"#features: {len(cols)}")

    Xtr = X_train_sig_core_32[cols]
    Xva = X_val_sig_core_32[cols]
    Xte = X_test_sig_core_32[cols]

    cat_model = CatBoostRegressor(**best_cb_params)

    cat_model.fit(
        Xtr,
        y_train_vec,
        eval_set=(Xva, y_val_vec),
        verbose=False,
    )

    pred_val  = cat_model.predict(Xva)
    pred_test = cat_model.predict(Xte)

    val_da  = directional_accuracy(y_val_vec,  pred_val)
    test_da = directional_accuracy(y_test_vec, pred_test)

    print(f"Val DA:  {val_da:.4f}")
    print(f"Test DA: {test_da:.4f}")

    cat_results.append({
        "model": f"CatBoost - {name}",
        "val_DA": val_da,
        "test_DA": test_da,
        "n_features": len(cols),
    })

cat_results_df = pd.DataFrame(cat_results).sort_values("val_DA", ascending=False)
cat_results_df


=== CatBoost on feature set: core_struct ===
#features: 151
Val DA:  0.4970
Test DA: 0.5198

=== CatBoost on feature set: struct_plus_top1 ===
#features: 26
Val DA:  0.4937
Test DA: 0.5190

=== CatBoost on feature set: dow_month_sector ===
#features: 25
Val DA:  0.5062
Test DA: 0.5246

=== CatBoost on feature set: signal_plus_struct ===
#features: 166
Val DA:  0.4961
Test DA: 0.5165


,model,val_DA,test_DA,n_features
2,CatBoost - dow_month_sector,0.506177,0.524554,25
0,CatBoost - core_struct,0.497014,0.519823,151
3,CatBoost - signal_plus_struct,0.496100,0.516480,166
1,CatBoost - struct_plus_top1,0.493663,0.519038,26


In [49]:
from catboost import CatBoostRegressor
from itertools import combinations
import numpy as np
import pandas as pd

# ============================================
# 1. Define feature groups from X_train_sig_core
# ============================================
# Assumes X_train_sig_core has all numeric / dummy features + maybe 'Date'/'tic'

all_cols = X_train_sig_core.columns.tolist()

# Basic groups (adapt names if needed)
sig_cols = [c for c in all_cols if c in [
    "ma_1m", "ma_2d", "ma_3d", "ma_4d", "ma_7d",
    "ma_1w", "ma_2w", "ma_14d", "ma_30d", "ma_50d",
    "ema_ema_5", "ema_ema_10", "ema_ema_20", "ema_ema_50",
    "b0", "s3_month_of_year_mean", "s4_dow_mean", "s7_roll63_mean_return",
    "Close"
] and c in all_cols]   # guard in case some are missing

vol_cols      = [c for c in all_cols if c == "log_volume"]
dow_cols      = [c for c in all_cols if c.startswith("day_of_week_")]
month_cols    = [c for c in all_cols if c.startswith("month_")]
sector_cols   = [c for c in all_cols if c.startswith("sector_")]
subsector_cols= [c for c in all_cols if c.startswith("subsector_")]

feature_groups = {
    "SIG":        sig_cols,
    "VOL":        vol_cols,
    "DOW":        dow_cols,
    "MONTH":      month_cols,
    "SECTOR":     sector_cols,
    "SUBSECTOR":  subsector_cols,
}

# Drop Date, keep tic as categorical if present
base_cols = [c for c in all_cols if c not in ["Date"]]

# ============================================
# 2. Extract best CatBoost hyperparams from tuning
# ============================================
best_cb = results_cb_df.iloc[0]  # assumes sorted by val_DA desc

best_cb_params = {
    "depth":             int(best_cb["depth"]),
    "learning_rate":     float(best_cb["learning_rate"]),
    "l2_leaf_reg":       float(best_cb["l2_leaf_reg"]),
    "bagging_temperature": float(best_cb["bagging_temperature"]),
    "iterations":        int(best_cb["best_iteration"]),  # from tuning with early stopping
    "loss_function":     "RMSE",
    "random_seed":       42,
    "verbose":           False,
}

# ============================================
# 3. Helper: build feature set & evaluate CatBoost
# ============================================
def run_catboost_on_feature_combo(group_names):
    """
    group_names: list like ["SIG", "DOW"] etc
    """
    # union of columns in those groups
    cols = []
    for g in group_names:
        cols.extend(feature_groups[g])
    # remove duplicates, keep only base cols
    cols = sorted(set(c for c in cols if c in base_cols))

    # Optionally include 'tic' as categorical if present
    cat_features_idx = []
    if "tic" in X_train_sig_core.columns:
        cols_with_tic = ["tic"] + cols  # put tic first (optional)
        cat_features_idx = [0]          # index of 'tic' in cols_with_tic
    else:
        cols_with_tic = cols

    Xtr = X_train_sig_core[cols_with_tic]
    Xva = X_val_sig_core[cols_with_tic]
    Xte = X_test_sig_core[cols_with_tic]

    model = CatBoostRegressor(
        **best_cb_params,
    )

    model.fit(
        Xtr,
        y_train_vec,
        eval_set=(Xva, y_val_vec),
        cat_features=cat_features_idx,
        verbose=False,
    )

    pred_val  = model.predict(Xva)
    pred_test = model.predict(Xte)

    val_da  = (np.sign(pred_val)  == np.sign(y_val_vec)).mean()
    test_da = (np.sign(pred_test) == np.sign(y_test_vec)).mean()

    combo_label = "+".join(group_names)
    print(f"{combo_label}:")
    print(f"  #features: {len(cols_with_tic)}")
    print(f"  Val DA:    {val_da:.4f}")
    print(f"  Test DA:   {test_da:.4f}")
    print()

    return {
        "groups": combo_label,
        "num_features": len(cols_with_tic),
        "val_DA": val_da,
        "test_DA": test_da,
    }

# ============================================
# 4. Enumerate group combinations & run CatBoost
# ============================================
results_cat_feature_combos = []

group_keys = list(feature_groups.keys())

# Control how crazy you want this to be
max_group_size = 3  # e.g., all 1-, 2-, 3-group combos

for r in range(1, max_group_size + 1):
    for combo in combinations(group_keys, r):
        res = run_catboost_on_feature_combo(list(combo))
        results_cat_feature_combos.append(res)

cat_feature_combo_df = pd.DataFrame(results_cat_feature_combos).sort_values(
    "val_DA", ascending=False
)
cat_feature_combo_df.head()

SIG:
  #features: 17
  Val DA:    0.4996
  Test DA:   0.5202

VOL:
  #features: 2
  Val DA:    0.4931
  Test DA:   0.5199

DOW:
  #features: 5
  Val DA:    0.4931
  Test DA:   0.5199

MONTH:
  #features: 12
  Val DA:    0.4931
  Test DA:   0.5199

SECTOR:
  #features: 11
  Val DA:    0.4931
  Test DA:   0.5199

SUBSECTOR:
  #features: 125
  Val DA:    0.4931
  Test DA:   0.5199

SIG+VOL:
  #features: 18
  Val DA:    0.4996
  Test DA:   0.5202

SIG+DOW:
  #features: 21
  Val DA:    0.5059
  Test DA:   0.5202

SIG+MONTH:
  #features: 28
  Val DA:    0.4931
  Test DA:   0.5199

SIG+SECTOR:
  #features: 27
  Val DA:    0.4998
  Test DA:   0.5202

SIG+SUBSECTOR:
  #features: 141
  Val DA:    0.5050
  Test DA:   0.5202

VOL+DOW:
  #features: 6
  Val DA:    0.4931
  Test DA:   0.5199

VOL+MONTH:
  #features: 13
  Val DA:    0.4931
  Test DA:   0.5199

VOL+SECTOR:
  #features: 12
  Val DA:    0.4931
  Test DA:   0.5199

VOL+SUBSECTOR:
  #features: 126
  Val DA:    0.4931
  Test DA:   0.5199

D

,groups,num_features,val_DA,test_DA
37,DOW+MONTH+SECTOR,26,0.517151,0.526422
38,DOW+MONTH+SUBSECTOR,140,0.513103,0.527280
15,DOW+MONTH,16,0.506433,0.523006
7,SIG+DOW,21,0.505856,0.520240
10,SIG+SUBSECTOR,141,0.505006,0.520176


Given best is DOW + Month + Sector, run model on this

In [54]:
# Start from the same df used for your catboost experiments
# (this should be the one that already has the one-hot cols)
X_train_sig_core   # <- whatever you've been using

# Identify the DOW / MONTH / SECTOR columns
dow_cols = [c for c in X_train_sig_core.columns if c.startswith("day_of_week_")]
month_cols = [c for c in X_train_sig_core.columns if c.startswith("month_")]
sector_cols = [c for c in X_train_sig_core.columns if c.startswith("sector_")]

dms_cols = dow_cols + month_cols + sector_cols

print("Num DOW cols:   ", len(dow_cols))
print("Num MONTH cols: ", len(month_cols))
print("Num SECTOR cols:", len(sector_cols))
print("Total DMS cols: ", len(dms_cols))

# Build train/val/test subsets
X_train_dms = X_train_sig_core[dms_cols].copy()
X_val_dms   = X_val_sig_core[dms_cols].copy()
X_test_dms  = X_test_sig_core[dms_cols].copy()

print("X_train_dms shape:", X_train_dms.shape)
print("X_val_dms shape:  ", X_val_dms.shape)
print("X_test_dms shape: ", X_test_dms.shape)

# Safety check – crash early if something is wrong
assert X_train_dms.shape[1] > 0, "No features found for DOW+MONTH+SECTOR!"

Num DOW cols:    4
Num MONTH cols:  11
Num SECTOR cols: 10
Total DMS cols:  25
X_train_dms shape: (729659, 25)
X_val_dms shape:   (124747, 25)
X_test_dms shape:  (124727, 25)


In [55]:
import numpy as np
from catboost import CatBoostRegressor

param_space = {
    "depth": [4, 6, 8, 10],
    "learning_rate": [0.01, 0.02, 0.03, 0.05],
    "l2_leaf_reg": [1, 3, 5, 7, 9],
    "bagging_temperature": [0, 1, 5],
    "border_count": [32, 64, 128],
}

def sample_params():
    return {
        "depth": np.random.choice(param_space["depth"]),
        "learning_rate": np.random.choice(param_space["learning_rate"]),
        "l2_leaf_reg": np.random.choice(param_space["l2_leaf_reg"]),
        "bagging_temperature": np.random.choice(param_space["bagging_temperature"]),
        "border_count": np.random.choice(param_space["border_count"]),
    }

results = []
N_TRIALS = 25

for i in range(N_TRIALS):
    params = sample_params()
    print(f"\n=== Trial {i+1}/{N_TRIALS} ===")
    print("Params:", params)

    model = CatBoostRegressor(
        loss_function="RMSE",
        iterations=3000,            # let early stopping cut it
        od_type="Iter",
        od_wait=50,                 # stop if no improvement
        random_seed=42,
        verbose=False,
        **params
    )

    model.fit(
        X_train_dms, y_train_vec,
        eval_set=(X_val_dms, y_val_vec),
        early_stopping_rounds=50,
        verbose=False
    )

    pred_val = model.predict(X_val_dms)
    val_da = directional_accuracy(y_val_vec, pred_val)

    print(f"Trial {i+1} Val DA: {val_da:.4f}")
    print(f"Best iteration: {model.get_best_iteration()}")

    results.append({
        **params,
        "val_DA": val_da,
        "best_iteration": model.get_best_iteration()
    })

results_cb_df = pd.DataFrame(results).sort_values("val_DA", ascending=False)
results_cb_df.head()


=== Trial 1/25 ===
Params: {'depth': np.int64(4), 'learning_rate': np.float64(0.03), 'l2_leaf_reg': np.int64(9), 'bagging_temperature': np.int64(0), 'border_count': np.int64(32)}
Trial 1 Val DA: 0.5151
Best iteration: 246

=== Trial 2/25 ===
Params: {'depth': np.int64(8), 'learning_rate': np.float64(0.01), 'l2_leaf_reg': np.int64(1), 'bagging_temperature': np.int64(5), 'border_count': np.int64(128)}
Trial 2 Val DA: 0.5145
Best iteration: 228

=== Trial 3/25 ===
Params: {'depth': np.int64(8), 'learning_rate': np.float64(0.01), 'l2_leaf_reg': np.int64(5), 'bagging_temperature': np.int64(5), 'border_count': np.int64(32)}
Trial 3 Val DA: 0.5159
Best iteration: 246

=== Trial 4/25 ===
Params: {'depth': np.int64(8), 'learning_rate': np.float64(0.01), 'l2_leaf_reg': np.int64(3), 'bagging_temperature': np.int64(5), 'border_count': np.int64(64)}
Trial 4 Val DA: 0.5145
Best iteration: 228

=== Trial 5/25 ===
Params: {'depth': np.int64(4), 'learning_rate': np.float64(0.05), 'l2_leaf_reg': np.int

,depth,learning_rate,l2_leaf_reg,bagging_temperature,border_count,val_DA,best_iteration
20,4,0.01,5,1,128,0.517744,597
23,6,0.03,1,0,32,0.517584,152
8,6,0.03,5,0,128,0.517584,152
12,4,0.05,9,5,128,0.516782,139
4,4,0.05,1,1,32,0.516782,139


In [56]:
best = results_cb_df.iloc[0]

best_params = {
    "depth": int(best["depth"]),
    "learning_rate": float(best["learning_rate"]),
    "l2_leaf_reg": float(best["l2_leaf_reg"]),
    "bagging_temperature": float(best["bagging_temperature"]),
    "border_count": int(best["border_count"]),
}

final_cb = CatBoostRegressor(
    loss_function="RMSE",
    iterations=int(best["best_iteration"]),
    random_seed=42,
    verbose=False,
    **best_params
)

final_cb.fit(X_train_dms, y_train_vec)

val_pred  = final_cb.predict(X_val_dms)
test_pred = final_cb.predict(X_test_dms)

print("Final CatBoost (DOW+MONTH+SECTOR):")
print("Val DA :", directional_accuracy(y_val_vec, val_pred))
print("Test DA:", directional_accuracy(y_test_vec, test_pred))

Final CatBoost (DOW+MONTH+SECTOR):
Val DA : 0.5177439136812909
Test DA: 0.5336374642218605


In [31]:
from catboost import CatBoostRegressor

# -----------------------------
# 1. Define DOW+MONTH+SECTOR columns if not already
# -----------------------------
# Assuming these were already created earlier:
# dow_cols, month_cols, sector_cols

dms_cols = dow_cols + month_cols + sector_cols  # 4 + 11 + 10 = 25

# X_train_sig_core_32, X_val_sig_core_32, X_test_sig_core_32
# should each contain DMS columns + other stuff, plus 'tic'

# Extract ONLY DMS numeric features
X_train_dms = X_train_sig_core_32[dms_cols].copy()
X_val_dms   = X_val_sig_core_32[dms_cols].copy()
X_test_dms  = X_test_sig_core_32[dms_cols].copy()

# -----------------------------
# 2. Add 'tic' as a categorical column
# -----------------------------
# Use the same index alignment as X_train_sig_core_32 etc.
X_train_dms_tic = X_train_dms.copy()
X_val_dms_tic   = X_val_dms.copy()
X_test_dms_tic  = X_test_dms.copy()

X_train_dms_tic["tic"] = X_train_sig_core["tic"].values
X_val_dms_tic["tic"]   = X_val_sig_core["tic"].values
X_test_dms_tic["tic"]  = X_test_sig_core["tic"].values

# Column index of 'tic' for CatBoost
cat_features = [X_train_dms_tic.columns.get_loc("tic")]

print("DMS+tic train shape:", X_train_dms_tic.shape)
print("Categorical feature indices:", cat_features)

# -----------------------------
# 3. Use best CatBoost hyperparameters (from your tuning)
# -----------------------------
best_depth            = 4
best_learning_rate    = 0.01
best_l2_leaf_reg      = 5
best_bagging_temp     = 1
best_border_count     = 128
best_iterations       = 597  # from best_iteration

cat_tic_model = CatBoostRegressor(
    loss_function="RMSE",
    depth=best_depth,
    learning_rate=best_learning_rate,
    l2_leaf_reg=best_l2_leaf_reg,
    bagging_temperature=best_bagging_temp,
    border_count=best_border_count,
    iterations=best_iterations,
    random_seed=42,
    verbose=False
)

# -----------------------------
# 4. Fit on DMS+tic and evaluate DA
# -----------------------------
cat_tic_model.fit(
    X_train_dms_tic,
    y_train_vec,
    eval_set=(X_val_dms_tic, y_val_vec),
    cat_features=cat_features,
    verbose=False,
)

pred_val_cat_tic  = cat_tic_model.predict(X_val_dms_tic)
pred_test_cat_tic = cat_tic_model.predict(X_test_dms_tic)

val_DA_cat_tic  = directional_accuracy(y_val_vec,  pred_val_cat_tic)
test_DA_cat_tic = directional_accuracy(y_test_vec, pred_test_cat_tic)

print("CatBoost (DOW+MONTH+SECTOR + tic):")
print("Val DA: ", val_DA_cat_tic)
print("Test DA:", test_DA_cat_tic)


DMS+tic train shape: (729659, 26)
Categorical feature indices: [25]
CatBoost (DOW+MONTH+SECTOR + tic):
Val DA:  0.5178962219532334
Test DA: 0.5354093339854241


In [32]:
import pandas as pd

# DOW + MONTH + SECTOR columns
dms_cols = dow_cols + month_cols + sector_cols

# Numeric DMS-only part
X_train_dms = X_train_sig_core[dms_cols].copy()
X_val_dms   = X_val_sig_core[dms_cols].copy()
X_test_dms  = X_test_sig_core[dms_cols].copy()

# Add tic as categorical feature
X_train_dmstic = X_train_dms.copy()
X_val_dmstic   = X_val_dms.copy()
X_test_dmstic  = X_test_dms.copy()

X_train_dmstic["tic"] = X_train_sig_core["tic"].astype("category")
X_val_dmstic["tic"]   = X_val_sig_core["tic"].astype("category")
X_test_dmstic["tic"]  = X_test_sig_core["tic"].astype("category")

print("DMS+tic train shape:", X_train_dmstic.shape)
print("DMS+tic val shape:  ", X_val_dmstic.shape)
print("DMS+tic test shape: ", X_test_dmstic.shape)

# index of categorical feature for CatBoost
cat_features = [X_train_dmstic.columns.get_loc("tic")]
print("Categorical feature indices:", cat_features)

DMS+tic train shape: (729659, 26)
DMS+tic val shape:   (124747, 26)
DMS+tic test shape:  (124727, 26)
Categorical feature indices: [25]


In [33]:
import numpy as np
from catboost import CatBoostRegressor

# Hyperparameter search space (centered around your good config)
param_space = {
    "depth":             [4, 6, 8],
    "learning_rate":     [0.005, 0.01, 0.02],
    "l2_leaf_reg":       [3, 5, 7],
    "bagging_temperature":[0, 1, 3],
    "border_count":      [64, 128],
}

def sample_params():
    return {
        "depth":             int(np.random.choice(param_space["depth"])),
        "learning_rate":     float(np.random.choice(param_space["learning_rate"])),
        "l2_leaf_reg":       float(np.random.choice(param_space["l2_leaf_reg"])),
        "bagging_temperature": float(np.random.choice(param_space["bagging_temperature"])),
        "border_count":      int(np.random.choice(param_space["border_count"])),
    }

cb_dmstic_results = []
N_TRIALS = 25  # you can bump to 40–50 if time allows

for i in range(1, N_TRIALS + 1):
    params = sample_params()
    print(f"\n=== CatBoost DMS+tic Trial {i}/{N_TRIALS} ===")
    print("Params:", params)

    model = CatBoostRegressor(
        loss_function="RMSE",
        iterations=3000,          # big cap; early stopping will cut it
        random_seed=42,
        verbose=False,
        **params
    )

    model.fit(
        X_train_dmstic,
        y_train_vec,
        eval_set=(X_val_dmstic, y_val_vec),
        cat_features=cat_features,
        early_stopping_rounds=100,
        verbose=False
    )

    best_iter = model.get_best_iteration()
    if best_iter is None or best_iter <= 0:
        best_iter = params["iterations"] if "iterations" in params else 3000

    pred_val = model.predict(X_val_dmstic)
    val_da = directional_accuracy(y_val_vec, pred_val)

    print(f"Val DA: {val_da:.4f} | best_iter: {best_iter}")

    cb_dmstic_results.append({
        **params,
        "val_DA": val_da,
        "best_iteration": best_iter,
    })

cb_dmstic_df = pd.DataFrame(cb_dmstic_results).sort_values("val_DA", ascending=False)
cb_dmstic_df.head()


=== CatBoost DMS+tic Trial 1/25 ===
Params: {'depth': 4, 'learning_rate': 0.02, 'l2_leaf_reg': 3.0, 'bagging_temperature': 3.0, 'border_count': 64}
Val DA: 0.5163 | best_iter: 335

=== CatBoost DMS+tic Trial 2/25 ===
Params: {'depth': 4, 'learning_rate': 0.005, 'l2_leaf_reg': 5.0, 'bagging_temperature': 3.0, 'border_count': 64}
Val DA: 0.5153 | best_iter: 1558

=== CatBoost DMS+tic Trial 3/25 ===
Params: {'depth': 4, 'learning_rate': 0.02, 'l2_leaf_reg': 7.0, 'bagging_temperature': 1.0, 'border_count': 64}
Val DA: 0.5156 | best_iter: 372

=== CatBoost DMS+tic Trial 4/25 ===
Params: {'depth': 4, 'learning_rate': 0.005, 'l2_leaf_reg': 5.0, 'bagging_temperature': 0.0, 'border_count': 64}
Val DA: 0.5153 | best_iter: 1558

=== CatBoost DMS+tic Trial 5/25 ===
Params: {'depth': 4, 'learning_rate': 0.02, 'l2_leaf_reg': 5.0, 'bagging_temperature': 1.0, 'border_count': 128}
Val DA: 0.5169 | best_iter: 405

=== CatBoost DMS+tic Trial 6/25 ===
Params: {'depth': 4, 'learning_rate': 0.02, 'l2_leaf_

,depth,learning_rate,l2_leaf_reg,bagging_temperature,border_count,val_DA,best_iteration
12,6,0.005,7.0,0.0,64,0.517560,937
18,6,0.005,7.0,0.0,128,0.517560,937
15,4,0.020,5.0,1.0,128,0.516886,405
4,4,0.020,5.0,1.0,128,0.516886,405
17,4,0.020,5.0,3.0,128,0.516886,405


In [34]:
# Pick best config
best_row = cb_dmstic_df.iloc[0]

best_cb_dmstic_params = {
    "depth":             int(best_row["depth"]),
    "learning_rate":     float(best_row["learning_rate"]),
    "l2_leaf_reg":       float(best_row["l2_leaf_reg"]),
    "bagging_temperature": float(best_row["bagging_temperature"]),
    "border_count":      int(best_row["border_count"]),
}

best_iter = int(best_row["best_iteration"])
print("Best DMS+tic params:", best_cb_dmstic_params)
print("Best iteration:", best_iter)

final_cb_dmstic = CatBoostRegressor(
    loss_function="RMSE",
    iterations=best_iter,   # use best_iteration from tuning
    random_seed=42,
    verbose=False,
    **best_cb_dmstic_params
)

final_cb_dmstic.fit(
    X_train_dmstic,
    y_train_vec,
    cat_features=cat_features,
    verbose=False
)

# Evaluate on val and test
val_pred_cb_dmstic  = final_cb_dmstic.predict(X_val_dmstic)
test_pred_cb_dmstic = final_cb_dmstic.predict(X_test_dmstic)

val_DA_cb_dmstic  = directional_accuracy(y_val_vec,  val_pred_cb_dmstic)
test_DA_cb_dmstic = directional_accuracy(y_test_vec, test_pred_cb_dmstic)

print("\nFinal CatBoost (DOW+MONTH+SECTOR + tic):")
print("Val DA: ", val_DA_cb_dmstic)
print("Test DA:", test_DA_cb_dmstic)

Best DMS+tic params: {'depth': 6, 'learning_rate': 0.005, 'l2_leaf_reg': 7.0, 'bagging_temperature': 0.0, 'border_count': 64}
Best iteration: 937

Final CatBoost (DOW+MONTH+SECTOR + tic):
Val DA:  0.5175595405099922
Test DA: 0.5272394910484498


#### Model 7 - Light GBM

In [57]:
import re

def make_lgbm_safe_colnames(cols):
    """
    Turn arbitrary column names into LightGBM-safe names:
    - replace any non-alphanumeric / underscore char with '_'
    """
    new_cols = []
    for c in cols:
        safe = re.sub(r'[^0-9A-Za-z_]+', '_', c)  # everything weird -> '_'
        new_cols.append(safe)
    return new_cols

# 1) Build a mapping from original -> safe names based on TRAIN columns
orig_cols = list(X_train_sig_core_m.columns)
safe_cols = make_lgbm_safe_colnames(orig_cols)

col_map = dict(zip(orig_cols, safe_cols))

# 2) Apply the same mapping to train/val/test
X_train_lgbm = X_train_sig_core_m.rename(columns=col_map)
X_val_lgbm   = X_val_sig_core_m.rename(columns=col_map)
X_test_lgbm  = X_test_sig_core_m.rename(columns=col_map)

In [58]:
from lightgbm import LGBMRegressor

lgbm = LGBMRegressor(
    n_estimators=1500,
    max_depth=-1,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
)

lgbm.fit(X_train_lgbm, y_train_vec)

pred_val_lgbm  = lgbm.predict(X_val_lgbm)
pred_test_lgbm = lgbm.predict(X_test_lgbm)

print("LightGBM (sig + core):")
print("Val DA: ", directional_accuracy(y_val_vec,  pred_val_lgbm))
print("Test DA:", directional_accuracy(y_test_vec, pred_test_lgbm))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.007861 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 4633
[LightGBM] [Info] Number of data points in the train set: 729659, number of used features: 166
[LightGBM] [Info] Start training from score 0.000880
LightGBM (sig + core):
Val DA:  0.4963566258106407
Test DA: 0.505600230904295


In [60]:
import re
import numpy as np
import pandas as pd
from itertools import product
from lightgbm import LGBMRegressor, early_stopping

# ---------- 1. Build DOW+MONTH+SECTOR matrices ----------

dow_cols    = [c for c in X_train_sig_core_32.columns if c.startswith("day_of_week_")]
month_cols  = [c for c in X_train_sig_core_32.columns if c.startswith("month_")]
sector_cols = [c for c in X_train_sig_core_32.columns if c.startswith("sector_")]

dms_cols = dow_cols + month_cols + sector_cols

X_train_dms = X_train_sig_core_32[dms_cols].copy()
X_val_dms   = X_val_sig_core_32[dms_cols].copy()
X_test_dms  = X_test_sig_core_32[dms_cols].copy()

print("Num DOW cols:   ", len(dow_cols))
print("Num MONTH cols: ", len(month_cols))
print("Num SECTOR cols:", len(sector_cols))
print("Total DMS cols: ", len(dms_cols))
print("X_train_dms shape:", X_train_dms.shape)
print("X_val_dms shape:  ", X_val_dms.shape)
print("X_test_dms shape: ", X_test_dms.shape)

# ---------- 2. Make LightGBM-safe column names ----------

def make_lgbm_safe_colnames(cols):
    """Replace non [0-9A-Za-z_] with '_'."""
    new_cols = []
    for c in cols:
        safe = re.sub(r'[^0-9A-Za-z_]+', '_', c)
        new_cols.append(safe)
    return new_cols

orig_dms_cols = list(X_train_dms.columns)
safe_dms_cols = make_lgbm_safe_colnames(orig_dms_cols)
dms_col_map   = dict(zip(orig_dms_cols, safe_dms_cols))

X_train_dms_lgb = X_train_dms.rename(columns=dms_col_map)
X_val_dms_lgb   = X_val_dms.rename(columns=dms_col_map)
X_test_dms_lgb  = X_test_dms.rename(columns=dms_col_map)

# ---------- 3. Random search space for LightGBM ----------

param_space_lgb = {
    "num_leaves":       [31, 63, 127],
    "max_depth":        [-1, 6, 10],
    "learning_rate":    [0.03, 0.05, 0.1],
    "n_estimators":     [500, 800, 1200],
    "min_data_in_leaf": [20, 50, 100],
    "feature_fraction": [0.7, 0.9],
    "bagging_fraction": [0.7, 0.9],
}

def sample_lgb_params():
    return {
        "num_leaves":       int(np.random.choice(param_space_lgb["num_leaves"])),
        "max_depth":        int(np.random.choice(param_space_lgb["max_depth"])),
        "learning_rate":    float(np.random.choice(param_space_lgb["learning_rate"])),
        "n_estimators":     int(np.random.choice(param_space_lgb["n_estimators"])),
        "min_data_in_leaf": int(np.random.choice(param_space_lgb["min_data_in_leaf"])),
        "feature_fraction": float(np.random.choice(param_space_lgb["feature_fraction"])),
        "bagging_fraction": float(np.random.choice(param_space_lgb["bagging_fraction"])),
        # bagging_freq is usually 1 when bagging_fraction < 1
        "bagging_freq":     1,
    }

results_lgb = []
N_TRIALS = 25  # you can tweak this

for i in range(N_TRIALS):
    params = sample_lgb_params()
    print(f"=== LightGBM DMS Trial {i+1}/{N_TRIALS} ===")
    print("Params:", params)

    model = LGBMRegressor(
        objective="regression",
        random_state=42,
        n_jobs=-1,
        **params,
    )

    # Early stopping via callbacks
    callbacks = [early_stopping(stopping_rounds=50, verbose=False)]

    model.fit(
        X_train_dms_lgb,
        y_train_vec,
        eval_set=[(X_val_dms_lgb, y_val_vec)],
        eval_metric="rmse",
        callbacks=callbacks,
    )

    best_iter = model.best_iteration_ if hasattr(model, "best_iteration_") else params["n_estimators"]

    # Evaluate DA using the best iteration
    val_pred = model.predict(X_val_dms_lgb, num_iteration=best_iter)
    val_da   = directional_accuracy(y_val_vec, val_pred)

    print(f"  -> best_iter: {best_iter}, val_DA={val_da:.4f}\n")

    results_lgb.append({
        **params,
        "best_iteration": best_iter,
        "val_DA": val_da,
    })

results_lgb_df = pd.DataFrame(results_lgb).sort_values("val_DA", ascending=False)
results_lgb_df.head()

Num DOW cols:    4
Num MONTH cols:  11
Num SECTOR cols: 10
Total DMS cols:  25
X_train_dms shape: (729659, 25)
X_val_dms shape:   (124747, 25)
X_test_dms shape:  (124727, 25)
=== LightGBM DMS Trial 1/25 ===
Params: {'num_leaves': 63, 'max_depth': 6, 'learning_rate': 0.03, 'n_estimators': 800, 'min_data_in_leaf': 50, 'feature_fraction': 0.9, 'bagging_fraction': 0.7, 'bagging_freq': 1}
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.7, subsample=1.0 will be ignored. Current value: bagging_fraction=0.7
[LightGBM] [Warning] min_data_in_leaf is set=50, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=50
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fract

,num_leaves,max_depth,learning_rate,n_estimators,min_data_in_leaf,feature_fraction,bagging_fraction,bagging_freq,best_iteration,val_DA
24,63,-1,0.03,500,100,0.9,0.9,1,51,0.519908
7,63,10,0.10,800,20,0.9,0.9,1,15,0.519323
13,31,-1,0.03,500,50,0.7,0.9,1,81,0.519002
11,63,6,0.10,800,20,0.7,0.7,1,53,0.517215
22,63,6,0.10,800,50,0.9,0.7,1,33,0.516846


In [61]:
best_lgb = results_lgb_df.iloc[0]

best_lgb_params = {
    "num_leaves":       int(best_lgb["num_leaves"]),
    "max_depth":        int(best_lgb["max_depth"]),
    "learning_rate":    float(best_lgb["learning_rate"]),
    "n_estimators":     int(best_lgb["n_estimators"]),
    "min_data_in_leaf": int(best_lgb["min_data_in_leaf"]),
    "feature_fraction": float(best_lgb["feature_fraction"]),
    "bagging_fraction": float(best_lgb["bagging_fraction"]),
    "bagging_freq":     1,
}

best_iter = int(best_lgb["best_iteration"])

final_lgb = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=-1,
    **best_lgb_params,
)

final_lgb.fit(
    X_train_dms_lgb,
    y_train_vec,
    eval_set=[(X_val_dms_lgb, y_val_vec)],
    eval_metric="rmse",
    callbacks=[early_stopping(stopping_rounds=50, verbose=False)],
)

val_pred_lgb  = final_lgb.predict(X_val_dms_lgb,  num_iteration=best_iter)
test_pred_lgb = final_lgb.predict(X_test_dms_lgb, num_iteration=best_iter)

val_DA_lgb  = directional_accuracy(y_val_vec,  val_pred_lgb)
test_DA_lgb = directional_accuracy(y_test_vec, test_pred_lgb)

print("LightGBM (DOW+MONTH+SECTOR):")
print("Val DA: ", val_DA_lgb)
print("Test DA:", test_DA_lgb)

[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Info] Auto-choosing row-wise multi-threadi

In [35]:
import pandas as pd
from lightgbm import LGBMRegressor, early_stopping

# 1) Rebuild DOW + MONTH + SECTOR columns (DMS)
dow_cols    = [c for c in X_train_sig_core_32.columns if c.startswith("day_of_week_")]
month_cols  = [c for c in X_train_sig_core_32.columns if c.startswith("month_")]
sector_cols = [c for c in X_train_sig_core_32.columns if c.startswith("sector_")]

dms_cols = dow_cols + month_cols + sector_cols

# 2) Create DMS + tic DataFrames (no scaling needed for LightGBM)
X_train_dms_tic = X_train_sig_core_32[dms_cols].copy()
X_val_dms_tic   = X_val_sig_core_32[dms_cols].copy()
X_test_dms_tic  = X_test_sig_core_32[dms_cols].copy()

# add 'tic' from the original (non-32) DataFrames
X_train_dms_tic["tic"] = X_train_sig_core["tic"].values
X_val_dms_tic["tic"]   = X_val_sig_core["tic"].values
X_test_dms_tic["tic"]  = X_test_sig_core["tic"].values

print("DMS+tic train shape:", X_train_dms_tic.shape)

# 3) Make sure 'tic' is a *categorical* column with consistent categories
all_tics = pd.concat([
    X_train_dms_tic["tic"],
    X_val_dms_tic["tic"],
    X_test_dms_tic["tic"],
], axis=0)

all_tics_unique = sorted(all_tics.unique())

cat_type = pd.api.types.CategoricalDtype(categories=all_tics_unique)

X_train_dms_tic["tic"] = X_train_dms_tic["tic"].astype(cat_type)
X_val_dms_tic["tic"]   = X_val_dms_tic["tic"].astype(cat_type)
X_test_dms_tic["tic"]  = X_test_dms_tic["tic"].astype(cat_type)

# LightGBM can use column name 'tic' as categorical_feature
categorical_features = ["tic"]

# 4) Best params from earlier tuning (on DMS only)
best_lgb_params = {
    "num_leaves":       63,
    "max_depth":        -1,
    "learning_rate":    0.03,
    "n_estimators":     500,
    "min_data_in_leaf": 100,
    "feature_fraction": 0.9,
    "bagging_fraction": 0.9,
    "bagging_freq":     1,
}

# 5) Fit LightGBM with early stopping, now including tic
lgbm_dms_tic = LGBMRegressor(
    objective="regression",
    random_state=42,
    n_jobs=-1,
    **best_lgb_params,
)

callbacks = [early_stopping(stopping_rounds=50, verbose=False)]

lgbm_dms_tic.fit(
    X_train_dms_tic,
    y_train_vec,
    eval_set=[(X_val_dms_tic, y_val_vec)],
    eval_metric="rmse",
    callbacks=callbacks,
    categorical_feature=categorical_features,
)

best_iter_tic = (
    lgbm_dms_tic.best_iteration_
    if hasattr(lgbm_dms_tic, "best_iteration_") and lgbm_dms_tic.best_iteration_ is not None
    else best_lgb_params["n_estimators"]
)

# 6) Evaluate Directional Accuracy on val and test
val_pred_lgb_tic  = lgbm_dms_tic.predict(X_val_dms_tic,  num_iteration=best_iter_tic)
test_pred_lgb_tic = lgbm_dms_tic.predict(X_test_dms_tic, num_iteration=best_iter_tic)

val_DA_lgb_tic  = directional_accuracy(y_val_vec,  val_pred_lgb_tic)
test_DA_lgb_tic = directional_accuracy(y_test_vec, test_pred_lgb_tic)

print("LightGBM (DOW+MONTH+SECTOR + tic):")
print("  best_iteration:", best_iter_tic)
print("  Val DA: ", val_DA_lgb_tic)
print("  Test DA:", test_DA_lgb_tic)

DMS+tic train shape: (729659, 26)
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9
[LightGBM] [Warning] feature_fraction is set=0.9, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.9
[LightGBM] [Warning] min_data_in_leaf is set=100, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=100
[LightGBM] [Warning] Categorical features with more bins than the configured maximum bin number found.
[LightGBM] [Warning] For categorical features, max_bin and max_bin_by_feature may be ignored with a large number of categories.
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] bagging_freq is set=1, subsample_freq=0 will be ignored. Current value: bagging_freq=1
[LightGBM] [Warning] bagging_fraction is set=0.9, subsample=1.0 will be ig

Now that we know that DOW + MONTH + SECTOR gives the best result, fit linear regression and finetune XGBoost on this

In [62]:
from sklearn.linear_model import LinearRegression

# Cast to float (just to be safe)
X_train_dms_lr = X_train_dms.astype("float64")
X_val_dms_lr   = X_val_dms.astype("float64")
X_test_dms_lr  = X_test_dms.astype("float64")

lr_dms = LinearRegression()
lr_dms.fit(X_train_dms_lr, y_train_vec)

pred_val_lr_dms  = lr_dms.predict(X_val_dms_lr)
pred_test_lr_dms = lr_dms.predict(X_test_dms_lr)

val_DA_lr_dms  = directional_accuracy(y_val_vec,  pred_val_lr_dms)
test_DA_lr_dms = directional_accuracy(y_test_vec, pred_test_lr_dms)

print("Linear Regression (DOW + MONTH + SECTOR):")
print("  Val DA: ", val_DA_lr_dms)
print("  Test DA:", test_DA_lr_dms)

Linear Regression (DOW + MONTH + SECTOR):
  Val DA:  0.4781116980769077
  Test DA: 0.5153334883385313


In [63]:
import xgboost as xgb
from itertools import product

# Use float32 for XGBoost
X_train_dms_32 = X_train_dms.astype("float32")
X_val_dms_32   = X_val_dms.astype("float32")
X_test_dms_32  = X_test_dms.astype("float32")

y_train_32 = np.asarray(y_train_vec, dtype="float32")
y_val_32   = np.asarray(y_val_vec,   dtype="float32")
y_test_32  = np.asarray(y_test_vec,  dtype="float32")

param_grid_dms = {
    "max_depth":        [3, 4, 5],
    "learning_rate":    [0.03, 0.05],
    "n_estimators":     [400, 800],
    "subsample":        [0.7, 0.9],
    "colsample_bytree": [0.7, 0.9],
}

results_dms = []

for max_depth, lr, n_estimators, subsample, colsample in product(
    param_grid_dms["max_depth"],
    param_grid_dms["learning_rate"],
    param_grid_dms["n_estimators"],
    param_grid_dms["subsample"],
    param_grid_dms["colsample_bytree"],
):
    params = {
        "max_depth":        max_depth,
        "learning_rate":    lr,
        "n_estimators":     n_estimators,
        "subsample":        subsample,
        "colsample_bytree": colsample,
    }

    model = xgb.XGBRegressor(
        **params,
        tree_method="hist",
        n_jobs=-1,
        random_state=42,
    )

    model.fit(X_train_dms_32, y_train_32, verbose=False)

    val_pred = model.predict(X_val_dms_32)
    val_da   = directional_accuracy(y_val_32, val_pred)

    results_dms.append({
        "max_depth":        max_depth,
        "learning_rate":    lr,
        "n_estimators":     n_estimators,
        "subsample":        subsample,
        "colsample_bytree": colsample,
        "val_DA":           val_da,
    })

results_dms_df = pd.DataFrame(results_dms).sort_values("val_DA", ascending=False)
results_dms_df.head()

,max_depth,learning_rate,n_estimators,subsample,colsample_bytree,val_DA
20,4,0.03,800,0.7,0.7,0.516918
13,3,0.05,800,0.7,0.9,0.516862
16,4,0.03,400,0.7,0.7,0.516710
12,3,0.05,800,0.7,0.7,0.516654
15,3,0.05,800,0.9,0.9,0.516437


In [64]:
best_dms = results_dms_df.iloc[0]
best_params_dms = {
    "max_depth":        int(best_dms["max_depth"]),
    "learning_rate":    float(best_dms["learning_rate"]),
    "n_estimators":     int(best_dms["n_estimators"]),
    "subsample":        float(best_dms["subsample"]),
    "colsample_bytree": float(best_dms["colsample_bytree"]),
}
print("Best DMS params:", best_params_dms)

best_xgb_dms = xgb.XGBRegressor(
    **best_params_dms,
    tree_method="hist",
    n_jobs=-1,
    random_state=42,
)

best_xgb_dms.fit(X_train_dms_32, y_train_32, verbose=False)

pred_val_xgb_dms  = best_xgb_dms.predict(X_val_dms_32)
pred_test_xgb_dms = best_xgb_dms.predict(X_test_dms_32)

val_DA_xgb_dms  = directional_accuracy(y_val_32,  pred_val_xgb_dms)
test_DA_xgb_dms = directional_accuracy(y_test_32, pred_test_xgb_dms)

print("XGBoost (tuned, DOW + MONTH + SECTOR):")
print("  Val DA: ", val_DA_xgb_dms)
print("  Test DA:", test_DA_xgb_dms)

Best DMS params: {'max_depth': 4, 'learning_rate': 0.03, 'n_estimators': 800, 'subsample': 0.7, 'colsample_bytree': 0.7}
XGBoost (tuned, DOW + MONTH + SECTOR):
  Val DA:  0.5169182425228663
  Test DA: 0.5280011545214749
